In [0]:
import json, os, time
from datetime import datetime, timedelta, timezone
import requests

BASE_URL = "https://api.openf1.org/v1"
LANDING = "/Volumes/f1/bronze/landing/openf1"
SESSION_ENDPOINTS = ["drivers", "session_result", "laps", "stints", "pit", "weather", "race_control"]
SETTLE_TIME = timedelta(hours=12)   # give OpenF1 time to finish publishing a session's data

_last_call = 0.0

def get_openf1(endpoint, **params):
    """Call one OpenF1 endpoint. Waits 2s between calls and retries on errors."""
    global _last_call
    for attempt in range(1, 6):
        wait = 2 - (time.time() - _last_call)
        if wait > 0:
            time.sleep(wait)
        _last_call = time.time()

        r = requests.get(f"{BASE_URL}/{endpoint}", params=params, timeout=30)
        if r.status_code == 404:
            return []
        if r.status_code == 429 or r.status_code >= 500:
            print(f"  {endpoint}: HTTP {r.status_code}, retry {attempt} in {10 * attempt}s")
            time.sleep(10 * attempt)
            continue
        r.raise_for_status()
        return r.json()

    raise RuntimeError(f"{endpoint} failed after 5 attempts")

In [0]:
def write_json(path, data):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f)

def session_is_landed(year, session_key):
    """race_control is fetched last, so if its file exists the whole session is done."""
    return os.path.exists(f"{LANDING}/race_control/year={year}/session_key={session_key}.json")

def land_session(session):
    year, key = session["year"], session["session_key"]
    for endpoint in SESSION_ENDPOINTS:
        data = get_openf1(endpoint, session_key=key)
        write_json(f"{LANDING}/{endpoint}/year={year}/session_key={key}.json", data)

def land_season_lists(year):
    """Small lists, overwritten every run (the pipeline re-reads them in full)."""
    for endpoint in ["meetings", "sessions"]:
        data = get_openf1(endpoint, year=year)
        write_json(f"{LANDING}/{endpoint}/year={year}/{endpoint}.json", data)
        print(f"{endpoint}: {len(data)} rows")

def land_starting_grids(year, races, cutoff):
    """One grid file per weekend; never save an empty grid, so it is retried next run."""
    meeting_keys = sorted({s["meeting_key"] for s in races
                           if datetime.fromisoformat(s["date_end"]) < cutoff})
    for mk in meeting_keys:
        path = f"{LANDING}/starting_grid/year={year}/meeting_key={mk}.json"
        if os.path.exists(path):
            continue
        grid = get_openf1("starting_grid", meeting_key=mk)
        if grid:
            write_json(path, grid)
            print(f"grid  meeting {mk}: {len(grid)} rows")
        else:
            print(f"grid  meeting {mk}: none yet, will retry next run")

def ingest_year(year):
    cutoff = datetime.now(timezone.utc) - SETTLE_TIME
    land_season_lists(year)

    races = get_openf1("sessions", year=year, session_type="Race")
    for s in races:
        label = f'{s["session_key"]} {s["location"]} - {s["session_name"]}'
        if datetime.fromisoformat(s["date_end"]) > cutoff:
            print(f"skip  {label} (not finished, or finished less than 12h ago)")
        elif session_is_landed(s["year"], s["session_key"]):
            print(f"skip  {label} (already landed)")
        else:
            land_session(s)
            print(f"done  {label}")

    land_starting_grids(year, races, cutoff)

In [0]:
dbutils.widgets.text("years", "", "Years (comma-separated, empty = current season)")
years_param = dbutils.widgets.get("years").strip()

years = [int(y) for y in years_param.split(",")] if years_param else [datetime.now(timezone.utc).year]

for year in years:
    print(f"\n===== {year} =====")
    ingest_year(year)